In [1]:
import numpy as np
from scipy.linalg import expm, eigh

# ==================================================
# Pauli matrices
# ==================================================

I = np.eye(2, dtype=complex)

sx = np.array([[0,1],
               [1,0]], dtype=complex)

sy = np.array([[0,-1j],
               [1j,0]], dtype=complex)

sz = np.array([[1,0],
               [0,-1]], dtype=complex)

# ==================================================
# Tensor product operator acting on qubit i
# ==================================================

def op_on_qubit(op, i, N):

    ops = []

    for q in range(N):
        if q == i:
            ops.append(op)
        else:
            ops.append(I)

    out = ops[0]

    for k in range(1, N):
        out = np.kron(out, ops[k])

    return out


def op_on_pair(op1, i, op2, j, N):

    ops = []

    for q in range(N):

        if q == i:
            ops.append(op1)

        elif q == j:
            ops.append(op2)

        else:
            ops.append(I)

    out = ops[0]

    for k in range(1, N):
        out = np.kron(out, ops[k])

    return out

# ==================================================
# System parameters
# ==================================================

N = 4

theta = 0.6

dm2 = 1.0
E   = 1.0

omega = dm2/(4.0*E)
def interaction_matrix(N, dm2, E):
    """
    Build Jij = (dm2 / 4E) * (1 - cos(theta_ij)),
    with theta_ij = arccos(0.9) * |i-j|/(N-1).
    """
    i = np.arange(N)
    dij = np.abs(i[:, None] - i[None, :])   # |i-j|
    theta = np.arccos(0.9) * dij / (N - 1)
    J = (dm2 / (4.0 * E)) * (1.0 - np.cos(theta))
    return J
J = interaction_matrix(N, dm2, E)
# ==================================================
# Build Hamiltonian
# ==================================================

H = np.zeros((2**N,2**N), dtype=complex)

# Vacuum term

for i in range(N):

    H += omega * (
        -np.cos(2*theta) * op_on_qubit(sz, i, N)
        +np.sin(2*theta) * op_on_qubit(sx, i, N)
    )

# Neutrino-neutrino term

for i in range(N):
    for j in range(i+1, N):

        H += J[i,j] * (
            op_on_pair(sx,i,sx,j,N)
            + op_on_pair(sy,i,sy,j,N)
            + op_on_pair(sz,i,sz,j,N)
        )

# ==================================================
# Initial state |0011>
# ==================================================

psi0 = np.zeros(2**N, dtype=complex)

psi0[int("0011",2)] = 1.0

# ==================================================
# Time evolution
# ==================================================

times = np.linspace(0,10,100)

states = []

for t in times:

    U = expm(-1j*H*t)

    psi = U @ psi0

    states.append(psi)

# ==================================================
# Flavour occupation numbers
# ==================================================

flavour_occ = []

for psi in states:

    occ = []

    for i in range(N):

        val = np.real(
            np.vdot(
                psi,
                op_on_qubit(sz,i,N) @ psi
            )
        )

        occ.append(val)

    flavour_occ.append(occ)

flavour_occ = np.array(flavour_occ)

# ==================================================
# Correlation matrix
# ==================================================

def correlation_matrix(psi):

    C = np.zeros((N,N))

    for i in range(N):

        zi = np.real(
            np.vdot(
                psi,
                op_on_qubit(sz,i,N) @ psi
            )
        )

        for j in range(N):

            zj = np.real(
                np.vdot(
                    psi,
                    op_on_qubit(sz,j,N) @ psi
                )
            )

            zij = np.real(
                np.vdot(
                    psi,
                    op_on_pair(sz,i,sz,j,N) @ psi
                )
            )

            C[i,j] = zij - zi*zj

    return C

# Example at final time

C_final = correlation_matrix(states[-1])

print(C_final)

[[ 0.14654399  0.17137903 -0.10931365 -0.23214443]
 [ 0.17137903  0.22476745 -0.02913529 -0.10931365]
 [-0.10931365 -0.02913529 -0.45753741  0.17137903]
 [-0.23214443 -0.10931365  0.17137903 -0.21016444]]


In [5]:
U_pmns = np.array([
    [np.cos(theta), np.sin(theta)],
    [-np.sin(theta), np.cos(theta)]
], dtype=complex)

U_mass = U_pmns

for _ in range(N-1):
    U_mass = np.kron(U_mass, U_pmns)

In [6]:
psi_mass = U_mass.conj().T @ psi
C_final_mass = correlation_matrix(psi_mass)

print(C_final_mass)

[[ 0.14182516  0.17404385 -0.10895912 -0.22711493]
 [ 0.17404385  0.22016703 -0.02927466 -0.10895912]
 [-0.10895912 -0.02927466 -0.43438853  0.17404385]
 [-0.22711493 -0.10895912  0.17404385 -0.20037596]]


In [7]:
### Instantaneous manybody Eigenbasis

evals, evecs = eigh(H)

coeffs = evecs.conj().T @ psi

probabilities = np.abs(coeffs)**2
IPR = np.sum(probabilities**2)

PR = 1.0 / IPR

print(PR)

5.885445542247184
